# eBPF Bottleneck Classifier v7 — Champion Model Only (Retrain Script)

This notebook trains **only** the final champion model from `bottleneck_diagnosis_v7.ipynb` —
no strategy comparison, no exploratory cells, no feature-importance/SHAP diagnostics.

**Champion: `Tuned (Baseline (no handling))`**
i.e. a `RandomForestClassifier` grid-searched over `n_estimators` / `max_depth` /
`min_samples_leaf`, trained on the **raw, unresampled** training split (no SMOTE /
Borderline-SMOTE / ADASYN, no `class_weight='balanced'`).

Reference numbers from the full v7 run (yours may vary slightly depending on the exact
CSV parts you load):

| Metric        | v7 binary champion | v6 3-class isolated High |
|---------------|--------------------|---------------------------|
| Precision     | 0.882              | 0.690                     |
| Recall        | 0.642              | 0.763                     |
| F1            | 0.743              | 0.730                     |
| PR-AUC        | 0.831              | —                         |

Pipeline stages kept (all identical to v7, needed to reproduce the exact feature set
and train/test split the champion was fit on):
1. Load + clean CSVs
2. Feature engineering (causal signals only)
3. Explicit allowlist (no data leakage)
4. Binary label from `HIGH_THRESHOLD_NS`
5. Frozen train/test split
6. Grid search on the unresampled baseline → champion
7. Evaluation + artifact export (`champion_model_v7.pkl`, `feature_cols_v7.json`, `label_thresholds_v7.json`)


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, f1_score, accuracy_score,
    precision_recall_curve, average_precision_score, roc_auc_score, make_scorer
)

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 60)

RANDOM_STATE = 42

## 1. Load data

In [2]:
# -- Load multiple CSV parts (manually named, no auto-discovery) --------------
CSV_PARTS = [
    'perf_metrics1.csv',
    'perf_metrics2.csv',
    # add more here as you collect them
]
# -------------------------------------------------------------------------------

dfs = []
for path in CSV_PARTS:
    if not os.path.exists(path):
        print(f"  skipping (not found): {path}")
        continue
    part = pd.read_csv(path)
    part['source_file'] = path
    dfs.append(part)
    print(f"  loaded {path:<20}: {len(part):>8} rows")

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nCombined shape: {df_raw.shape}")
print(f"Sessions found : {df_raw['session_label'].unique().tolist()}")
df_raw['session_label'].value_counts()

  loaded perf_metrics1.csv   :   526895 rows
  loaded perf_metrics2.csv   :   250683 rows

Combined shape: (777578, 45)
Sessions found : ['idle', 'cpu_low', 'cpu_medium', 'cpu_high', 'cpu_overloaded', 'mem_low', 'mem_high', 'io_low', 'lock_low', 'mixed_load', 'cpu_extreme', 'cpu_ultra', 'mem_extreme', 'mem_thrashing', 'cpu_mem_extreme', 'cpu_mem_ultra', 'io_extreme', 'cpu_io_extreme', 'lock_extreme', 'lock_cpu_mix', 'everything_max', 'everything_insane', 'cpu_2x', 'cpu_4x', 'cpu_8x', 'mem_reclaim', 'mem_heavy_reclaim', 'cpu_mem_2x', 'cpu_mem_4x', 'cpu_mem_8x', 'cpu_mem_io', 'cpu_mem_io_heavy', 'lock_heavy', 'lock_cpu_heavy', 'everything_extreme', 'compile_only', 'compile_sysbench', 'compile_blender', 'full_system', 'mem_swap_thrashing', 'cpu_l3_cache_misses', 'mem_swap_thrash', 'mem_tlb_cache_miss', 'io_async_saturation', 'ctx_flood', 'llm_inference', 'ml_training', 'stream_memory_contention', 'cache_contention', 'context_switch_contention']


session_label
stream_memory_contention     146855
cache_contention             133428
context_switch_contention    121529
llm_inference                 39621
everything_max                21993
ml_training                   21422
everything_insane             20568
lock_cpu_heavy                11752
cpu_8x                        10213
lock_low                      10112
lock_heavy                     9933
everything_extreme             9705
cpu_mem_io                     9449
cpu_medium                     9261
full_system                    8964
cpu_low                        8331
cpu_mem_io_heavy               8082
cpu_mem_4x                     8024
cpu_mem_8x                     7969
mem_low                        7914
mem_extreme                    7764
mem_reclaim                    7449
compile_only                   6823
cpu_mem_2x                     6718
io_low                         6653
compile_sysbench               6585
compile_blender                6570
mem_tlb_cache_

## 2. Clean + engineer features (identical to v7/v6)

In [3]:
# -- CLEANING (same rules as v6/v7) ---------------------------------------------
df = df_raw.copy()
before = len(df)

df = df[df['pid'] > 0]
df = df[df['ctx_switches'] > 0]
df = df[df['latency_count'] > 0]

print(f"Cleaned: {before} -> {len(df)} rows ({before - len(df)} removed)")

Cleaned: 777578 -> 664256 rows (113322 removed)


In [4]:
# -- FEATURE ENGINEERING (causal signals only -- same as v6/v7) ----------------
df['involuntary_ratio']  = df['involuntary_switches'] / df['ctx_switches'].clip(lower=1)
df['voluntary_ratio']    = df['voluntary_switches']   / df['ctx_switches'].clip(lower=1)
df['runtime_per_switch'] = df['total_runtime_ns']     / df['ctx_switches'].clip(lower=1)
df['read_write_ratio']   = df['read_bytes'] / (df['write_bytes'] + 1)
df['io_bytes_total']     = df['read_bytes'] + df['write_bytes']
df['alloc_pressure']     = df['total_alloc_bytes'] / df['ctx_switches'].clip(lower=1)
df['lock_pressure']      = (
    df['mutex_contentions'] + df['rwsem_read_contentions'] + df['rwsem_write_contentions']
)

log_cols = ['total_runtime_ns', 'avg_syscall_latency_ns', 'max_syscall_latency_ns',
            'io_bytes_total', 'total_alloc_bytes']
for col in log_cols:
    if col in df.columns:
        df[f'log_{col}'] = np.log1p(df[col])

print(f"Features after engineering: {df.shape[1]} columns")

Features after engineering: 57 columns


In [5]:
# -- EXPLICIT ALLOWLIST (no data leakage) -- identical to v6/v7 ----------------
LEAKY_COLS = {
    'avg_stall_ns', 'stall_ns', 'max_stall_ns', 'latency_count',
    'stall_per_switch', 'log_stall_ns', 'log_avg_stall_ns', 'log_max_stall_ns',
    'stall_spike', 'log_stall_spike',
}

ALLOWED_FEATURES = [
    'ctx_switches', 'cpu_migrations', 'involuntary_switches', 'voluntary_switches',
    'involuntary_ratio', 'voluntary_ratio', 'avg_runq_ratio', 'runtime_per_switch',
    'minor_faults', 'major_faults', 'kmalloc_count', 'kfree_count',
    'total_alloc_bytes', 'alloc_pressure',
    'read_count', 'write_count', 'read_bytes', 'write_bytes', 'read_write_ratio', 'io_bytes_total',
    'mutex_contentions', 'avg_mutex_wait_ns', 'rwsem_read_contentions', 'rwsem_write_contentions', 'lock_pressure',
    'syscall_count', 'avg_syscall_latency_ns', 'futex_count',
    'log_total_runtime_ns', 'log_avg_syscall_latency_ns', 'log_max_syscall_latency_ns',
    'log_io_bytes_total', 'log_total_alloc_bytes',
]

FEATURE_COLS = [f for f in ALLOWED_FEATURES if f in df.columns]

leakage_check = set(FEATURE_COLS) & LEAKY_COLS
assert len(leakage_check) == 0, f"LEAKAGE DETECTED: {leakage_check}"
print(f"Leakage check passed -- {len(FEATURE_COLS)} features, none target-derived")

Leakage check passed -- 33 features, none target-derived


## 3. Binary label (High vs. Not-High)

In [6]:
HIGH_THRESHOLD_NS = 1_500_000   # 1.5 ms -- fixed value used for the champion run

df['y'] = (df['avg_stall_ns'] >= HIGH_THRESHOLD_NS).astype(int)
class_names_map = {0: 'Not-High', 1: 'High'}
TARGET_NAMES  = ['Not-High', 'High']
TARGET_LABELS = [0, 1]

print(f"Fixed threshold (ns): High >= {HIGH_THRESHOLD_NS:,}\n")
for cls in TARGET_LABELS:
    count = (df['y'] == cls).sum()
    pct = 100 * count / len(df)
    print(f"  Class {cls} ({class_names_map[cls]:<9}): {count:7d} rows  ({pct:5.1f}%)")

Fixed threshold (ns): High >= 1,500,000

  Class 0 (Not-High ):  635452 rows  ( 95.7%)
  Class 1 (High     ):   28804 rows  (  4.3%)


## 4. Frozen train/test split

In [7]:
X = df[FEATURE_COLS].fillna(0)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows (frozen)\n")
print("Class distribution -- Train:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(1).to_string())
print("\nClass distribution -- Test:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(1).to_string())

Train: 531404 rows | Test: 132852 rows (frozen)

Class distribution -- Train:
y
0   95.700
1    4.300

Class distribution -- Test:
y
0   95.700
1    4.300


## 5. Train the champion: grid search on the unresampled baseline

The champion strategy is **`Baseline (no handling)`** — no SMOTE/Borderline-SMOTE/ADASYN
resampling and no `class_weight='balanced'`. So the grid search below is fit directly on
`X_train, y_train` with `class_weight=None`, exactly reproducing how `tuned_model` was
produced in the v7 notebook when `best_strategy == 'Baseline (no handling)'`.

In [8]:
def evaluate_predictions(y_true, y_pred, y_proba, name):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=TARGET_LABELS, zero_division=0
    )
    macro_f1 = f1.mean()
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    result = {
        'strategy': name,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'precision_high': prec[1],
        'recall_high': rec[1],
        'f1_high': f1[1],
    }
    if y_proba is not None:
        result['pr_auc_high'] = average_precision_score(y_true, y_proba)
        result['roc_auc_high'] = roc_auc_score(y_true, y_proba)
    return result

In [9]:
f1_high_scorer = make_scorer(f1_score, pos_label=1, zero_division=0)

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 12, 20],
    'min_samples_leaf': [1, 5, 10],
}

# Champion strategy = 'Baseline (no handling)' -> no resampling, no class_weight
X_train_search, y_train_search = X_train, y_train
class_weight = None

base_rf = RandomForestClassifier(class_weight=class_weight, n_jobs=4, random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    base_rf, param_grid, scoring=f1_high_scorer, cv=3, n_jobs=1, refit=True
)
grid_search.fit(X_train_search, y_train_search)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV High-class F1: {grid_search.best_score_:.3f}")

champion_model = grid_search.best_estimator_
champion_pred = champion_model.predict(X_test)
champion_proba = champion_model.predict_proba(X_test)[:, 1]
champion_result = evaluate_predictions(y_test, champion_pred, champion_proba, 'Tuned (Baseline (no handling))')

print(f"\nChampion on frozen test set: precision={champion_result['precision_high']:.3f}, "
      f"recall={champion_result['recall_high']:.3f}, F1={champion_result['f1_high']:.3f}, "
      f"PR-AUC={champion_result['pr_auc_high']:.3f}")

del X_train_search, y_train_search; gc.collect()

Best params: {'max_depth': 20, 'min_samples_leaf': 1, 'n_estimators': 500}
Best CV High-class F1: 0.737

Champion on frozen test set: precision=0.882, recall=0.642, F1=0.743, PR-AUC=0.831


471

## 6. Deployment decision threshold + final report

In [10]:
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, champion_proba)
ap = average_precision_score(y_test, champion_proba)
roc_auc = roc_auc_score(y_test, champion_proba)
print(f"PR-AUC: {ap:.3f}   ROC-AUC: {roc_auc:.3f}\n")

print(f"{'threshold':>10}  {'precision':>10}  {'recall':>8}")
for target_prec in [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]:
    hits = np.where(prec_arr[:-1] >= target_prec)[0]
    if len(hits) == 0:
        continue
    idx = hits[0]
    print(f"{thresh_arr[idx]:>10.3f}  {prec_arr[idx]:>10.3f}  {rec_arr[idx]:>8.3f}")

# -- Set the deployment threshold here, informed by the table above -----------
DECISION_THRESHOLD = 0.5   # change based on the precision/recall trade-off printed above
y_pred_final = (champion_proba >= DECISION_THRESHOLD).astype(int)

print(f"\nUsing DECISION_THRESHOLD = {DECISION_THRESHOLD}")
print(classification_report(y_test, y_pred_final, labels=TARGET_LABELS, target_names=TARGET_NAMES, zero_division=0))

cm = confusion_matrix(y_test, y_pred_final, labels=TARGET_LABELS)
print("Confusion matrix [rows=true, cols=pred], labels=['Not-High','High']:")
print(cm)

PR-AUC: 0.831   ROC-AUC: 0.975

 threshold   precision    recall
     0.136       0.600     0.825
     0.218       0.700     0.783
     0.279       0.750     0.754
     0.340       0.800     0.717
     0.417       0.850     0.676
     0.551       0.900     0.621

Using DECISION_THRESHOLD = 0.5
              precision    recall  f1-score   support

    Not-High       0.98      1.00      0.99    127091
        High       0.88      0.64      0.74      5761

    accuracy                           0.98    132852
   macro avg       0.93      0.82      0.87    132852
weighted avg       0.98      0.98      0.98    132852

Confusion matrix [rows=true, cols=pred], labels=['Not-High','High']:
[[126597    494]
 [  2060   3701]]


## 7. Save artifacts

In [11]:
import joblib

joblib.dump(champion_model, 'champion_model_v7.pkl')
print("Saved: champion_model_v7.pkl")

with open('feature_cols_v7.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print("Saved: feature_cols_v7.json")

with open('label_thresholds_v7.json', 'w') as f:
    json.dump(
        {
            'high_threshold_ns': HIGH_THRESHOLD_NS,
            'decision_probability_threshold': DECISION_THRESHOLD,
        },
        f, indent=2
    )
print("Saved: label_thresholds_v7.json  (binary High cutoff + deployment probability threshold)")

pd.DataFrame([champion_result]).set_index('strategy').round(3).to_csv('champion_metrics_v7.csv')
print("Saved: champion_metrics_v7.csv")

Saved: champion_model_v7.pkl
Saved: feature_cols_v7.json
Saved: label_thresholds_v7.json  (binary High cutoff + deployment probability threshold)
Saved: champion_metrics_v7.csv


## Notes

- This is a **retrain script**, not a from-scratch redesign: `HIGH_THRESHOLD_NS`,
  `ALLOWED_FEATURES`, the 80/20 stratified split, and the grid are all copied verbatim
  from `bottleneck_diagnosis_v7.ipynb` so the resulting model is the same champion, not
  a new candidate.
- If your exact reported numbers (precision 0.882 / recall 0.642 / F1 0.743 / PR-AUC 0.831)
  don't reproduce bit-for-bit, the usual culprit is a different `CSV_PARTS` list (different
  amount/composition of data) — the split and hyperparameter search are both seeded with
  `RANDOM_STATE = 42`, so given the same input rows the result should match.
- `DECISION_THRESHOLD` is left at `0.5` as in the original notebook — re-run the
  precision/recall table above and adjust it if you want a different operating point for
  deployment (e.g. eBPF real-time predictor).